In [ ]:
## 2026.09.03 Xenium CRC twin of CODEX_hcc/HCC_train_validate_cv_UNIlabel_all.ipynb
## Cross-dataset training on P1CRC + P2CRC + P5CRC (transferred RCTD)
##
## CLI equivalent:
##   conda run --no-capture-output -n SeededNTM python -u \
##     code/Xenium_crc/CRC_train_validate_cv_UNIlabel.py --mode cross-dataset \
##     --use-spatial-context --spatial-k 8 --spatial-mode mean \
##     --pooled-save-result result_all_spatial
##
## Hierarchy (three-head):
##   L2  = final_CT          (fine / celltype_level2)
##   L12 = final_sublineage  (intermediate / celltype_level1)
##   L1  = final_lineage     (coarse / celltype_level0)
##
## Preprocess CSVs first:
##   conda run -n SeededNTM python code/Xenium_crc/match_xenium_cells_with_pixel.py
## Env: conda activate SeededNTM
## Set SKIP_POOLED_TRAIN=True in the config cell to reuse CLI weights (False = train).


In [ ]:
## CRC_train_validate_cv_UNIlabel_all.ipynb
## Pool Cases/{sample}/{sample}_matched_features.h5ad → dataset-level CV
## Outputs under: data/Xemium/CRC/Results/{save_result}/
## StarDist per-sample plots: .../{save_result}/stardist/{sample}/
## Show three levels: L2 / L12 / L1 prediction results


In [ ]:
import os

import warnings

# Keep notebook output clean.
warnings.filterwarnings("ignore")

# Choose physical GPU (string). Restart kernel after changing.
os.environ.setdefault("NCRT_CUDA_DEVICE", "0")
# Runtime env (GPU pin + tmpdir) before torch import.
# Ensure Hist2Pheno_pkg is importable before importing runtime helpers.
import sys as _sys
_pkg_dir = "/home/lingyu/ssd2/Python/Hist2Pheno/code/Hist2Pheno_pkg"
if _pkg_dir not in _sys.path:
    _sys.path.insert(0, _pkg_dir)

from runtime_env import configure_notebook_runtime
_rt_env = configure_notebook_runtime()
NCRT_PHYSICAL_GPU = _rt_env["physical_gpu"]


In [ ]:

import sys
import importlib
import pandas as pd
from pathlib import Path

path = "/home/lingyu/ssd2/Python/"
for _d in (
    f"{path}Hist2Pheno/code/Hist2Pheno_pkg",
    f"{path}Hist2Pheno/code/Xenium_crc",
):
    if _d not in sys.path:
        sys.path.insert(0, _d)

import base
import plot
import model as model_pkg
importlib.reload(base)
importlib.reload(plot)
importlib.reload(model_pkg)

import uni_label_cv_helpers as uni_nb
importlib.reload(uni_nb)

import CRC_train_validate_cv_UNIlabel as crc_cli
importlib.reload(crc_cli)

print("Imports OK")
print(f"CUDA_VISIBLE_DEVICES={os.environ.get('CUDA_VISIBLE_DEVICES')} (physical GPU {NCRT_PHYSICAL_GPU})")


In [ ]:
## CUDA/CPU sanity check + reproducibility (delegated to Hist2Pheno_pkg)
import torch
import random
import numpy as np
from torch_runtime import configure_torch_runtime

SEED = 42
_rt = configure_torch_runtime(seed=SEED)

# Keep the same variable names used downstream.
device = _rt.device
amp_scaler = _rt.amp_scaler
CUDA_DEVICE_INDEX = _rt.cuda_device_index
PHYSICAL_GPU = _rt.physical_gpu


## Cross-dataset training (Xenium CRC P1CRC + P2CRC + P5CRC)

Pool pre-built `{sample}_matched_features.h5ad`, run **dataset-level 5-fold CV** (~70% train / ~30% test; with 2 samples this is leave-one-replicate-out), then StarDist on **each** replicate for **three heads** (L2 / L12 / L1).

**Outputs:** `data/Xemium/CRC/Results/{save_result}/`

- Internal validation (OOF): L2 + L1 + L12 confusion matrices
- StarDist: `Results/{save_result}/stardist/{sample}/`

Run cells **in order** after kernel restart.

Set `QUICK_VALIDATE = True` in the config cell for a 1-epoch smoke test.
Set `SKIP_POOLED_TRAIN = True` to load an existing CLI checkpoint without retraining.


In [ ]:
from uni_label_cv_helpers import (
    RESULT_FIG,
    make_pooled_result_fig,
    make_pooled_stardist_result_fig,
    _auroc_csv_path_from_metrics,
    ensure_lp_extra_insample_preds,
    plot_he_confusion_matrices,
)
from model import predict_all_label_heads
from plot import plot_confusion_matrix

CRC_ROOT = Path(f"{path}Hist2Pheno/data/Xemium/CRC")
DATA_ROOT = CRC_ROOT / "Results"
CASES_ROOT = CRC_ROOT / "Cases"
save_result = "result_all_spatial"

# Quick smoke test: 2-fold, 1 epoch
QUICK_VALIDATE = False
# True → load CLI best_mlp_gpu.pt only (no fold resume, no all-sample refit).
SKIP_POOLED_TRAIN = False
VALIDATION_SUBSET = ["P1CRC", "P2CRC", "P5CRC"]

all_samples = crc_cli.discover_crc_samples(CASES_ROOT, require_h5ad=False, require_cells_csv=True)
_acq_map = crc_cli.load_he_acq_map(CASES_ROOT)
_target = VALIDATION_SUBSET if QUICK_VALIDATE else all_samples
for _s in _target:
    _h5 = CASES_ROOT / _s / f"{_s}_matched_features.h5ad"
    if _h5.is_file():
        continue
    print(f"Building HE h5ad for {_s} ...", flush=True)
    _ctx = crc_cli.RunContext(
        sample=_s,
        cases_root=CASES_ROOT,
        python_root=Path(path),
        therapy_data=_s,
        therapy_model=crc_cli.DEFAULT_THERAPY_MODEL,
        save_result="result",
        device=device,
        seed=SEED,
        match_tolerance=1.0,
        column_rename=dict(crc_cli.HCC_COLUMN_RENAME),
        force_rebuild_h5ad=False,
        input_dim=None,
        hidden_dims=(1024, 512, 256),
        cv_k=5,
        stratify_target="joint",
        patience=10,
        max_epochs=50,
        train_batch_size=4096,
        resume_from_checkpoints=True,
        ablation_tag="D_emph_L2_spatial_crc",
        hce_w1=1.0, hce_w2=2.0, hce_w12=1.0, hce_w_l12head=1.0, hce_w_l3=1.0, hce_w_l4=1.0,
        build_stardist_h5ad=False,
        acq_id=_acq_map[_s],
        use_spatial_context=True,
        spatial_k=8,
        spatial_mode="mean",
    )
    crc_cli.step_he_h5ad(_ctx)

POOLED_SAMPLES = crc_cli.discover_crc_samples(
    CASES_ROOT,
    sample=None,
    require_h5ad=True,
)
if QUICK_VALIDATE:
    POOLED_SAMPLES = [s for s in VALIDATION_SUBSET if s in POOLED_SAMPLES]
print(f"Cross-dataset: {len(POOLED_SAMPLES)} datasets (CRC three-head)")

pooled_result_fig, pooled_result_dir = make_pooled_result_fig(DATA_ROOT, save_result)
FIG = RESULT_FIG
print(f"Results → {pooled_result_dir}")

POOLED_CV_K = 2 if QUICK_VALIDATE else min(5, max(len(POOLED_SAMPLES), 1))  # 3 samples → group CV; deploy refits on all samples when n_sections < k
POOLED_MAX_EPOCHS = 1 if QUICK_VALIDATE else 50
POOLED_PATIENCE = 1 if QUICK_VALIDATE else 10
POOLED_TRAIN_GROUP_FRAC = 0.7
POOLED_TRAIN_BATCH_SIZE = 256 if QUICK_VALIDATE else 4096
POOLED_USE_SPATIAL = True
POOLED_SPATIAL_K = 8
POOLED_SPATIAL_MODE = "mean"
# Must match CLI --ablation-tag (demo.sh uses D_emph_L2_spatial_crc).
if POOLED_USE_SPATIAL and not QUICK_VALIDATE:
    POOLED_ABLATION_TAG = "D_emph_L2_spatial_crc"
elif QUICK_VALIDATE:
    POOLED_ABLATION_TAG = "D_emph_L2_smoke"
else:
    POOLED_ABLATION_TAG = "D_emph_L2"
POOLED_HIDDEN_DIMS = (1024, 512, 256)
POOLED_HCE = dict(hce_w1=1.0, hce_w2=2.0, hce_w12=1.0, hce_w_l12head=1.0, hce_w_l3=1.0, hce_w_l4=1.0)
POOLED_VAL_SELECTION = crc_cli.DEFAULT_VAL_SELECTION_METRIC  # three_tier_auc_sum
POOLED_CV_SELECTION = crc_cli.DEFAULT_CV_SELECTION_METRIC

pooled_ctx = crc_cli.PooledRunContext(
    data_root=DATA_ROOT,
    cases_root=CASES_ROOT,
    python_root=Path(path),
    samples=POOLED_SAMPLES,
    device=device,
    seed=SEED,
    save_result=save_result,
    input_dim=None,
    hidden_dims=POOLED_HIDDEN_DIMS,
    cv_k=POOLED_CV_K,
    train_group_frac=POOLED_TRAIN_GROUP_FRAC,
    patience=POOLED_PATIENCE,
    max_epochs=POOLED_MAX_EPOCHS,
    train_batch_size=POOLED_TRAIN_BATCH_SIZE,
    resume_from_checkpoints=not QUICK_VALIDATE,
    ablation_tag=POOLED_ABLATION_TAG,
    val_selection_metric=POOLED_VAL_SELECTION,
    cv_selection_metric=POOLED_CV_SELECTION,
    use_spatial_context=POOLED_USE_SPATIAL,
    spatial_k=POOLED_SPATIAL_K,
    spatial_mode=POOLED_SPATIAL_MODE,
    **POOLED_HCE,
)

print(f"Selection: val={POOLED_VAL_SELECTION}, cv={POOLED_CV_SELECTION}")
print(f"Ablation tag: {POOLED_ABLATION_TAG}, batch_size={POOLED_TRAIN_BATCH_SIZE}")
if POOLED_USE_SPATIAL:
    print(f"Spatial context: k={POOLED_SPATIAL_K}, mode={POOLED_SPATIAL_MODE!r}")
print("Three-head levels: L2=final_CT | L12=final_sublineage | L1=final_lineage")
print(f"SKIP_POOLED_TRAIN={SKIP_POOLED_TRAIN}")


## 1. Load pooled h5ad & prep CV arrays


In [ ]:
crc_cli.step_pooled_prepare(pooled_ctx)

cv_data = pooled_ctx.g["cv_data"]
scaler = pooled_ctx.g["scaler"]
class_names = pooled_ctx.g["class_names"]
class_names_level1 = pooled_ctx.g["class_names_level1"]
class_names_level12 = pooled_ctx.g.get("class_names_level12")
input_dim = pooled_ctx.g["input_dim"]

print(f"Pooled: {cv_data['X_f'].shape[0]:,} cells × {input_dim}")
print(f"L2 classes: {len(class_names)} | L12: {len(class_names_level12) if class_names_level12 is not None else 0} | L1: {len(class_names_level1)}")
print(f"datasets: {len(set(cv_data['groups_f']))}")
print(f"USE_THREE_HEAD={pooled_ctx.g.get('USE_THREE_HEAD')}")


## 2. Group K-fold training

Dataset-level CV: each fold trains on ~70% datasets, validates on held-out ~30%.

Checkpoint selection: `three_tier_auc_sum` (L2 + L1 + L12 macro AUROC).


In [ ]:
if SKIP_POOLED_TRAIN:
    crc_cli._ensure_pooled_inference_ready(pooled_ctx, require_train_if_missing=False)
    print("SKIP_POOLED_TRAIN: loaded existing best_mlp_gpu.pt (no retrain / no refit).")
else:
    crc_cli.step_pooled_train(pooled_ctx)

LP = pooled_ctx.g["LP"] if "LP" in pooled_ctx.g else {}
model = pooled_ctx.g.get("model")
BEST_MLP_CHECKPOINT = pooled_ctx.g["BEST_MLP_CHECKPOINT"]
print(f"Best checkpoint: {BEST_MLP_CHECKPOINT}")
if isinstance(LP, dict) and LP.get("best_fold"):
    print(f"Best fold: {LP['best_fold'].get('fold')}")


## 3. Internal validation (OOF) — three levels

- **L2**: OOF confusion + macro AUROC (from group CV)
- **L1 / L12**: in-sample predictions from the best-fold three-head model (with spatial kNN if enabled)


In [ ]:
# L2 OOF needs group-CV LP. CLI-only load has no val_preds — skip OOF.
if pooled_ctx.g.get("val_preds") is not None:
    crc_cli.step_pooled_he_validate(pooled_ctx)
else:
    print("SKIP_POOLED_TRAIN: skip OOF HE validate (no LP). Using loaded checkpoint for in-sample plots.")

# Extra: L1 + L12 confusion from the loaded deployment model on all pooled cells
g = pooled_ctx.g
_nbr = g.get("spatial_neighbor_index")
ensure_lp_extra_insample_preds(
    g["model"],
    scaler,
    cv_data,
    device,
    predict_all_label_heads,
    g,
    neighbor_index=_nbr,
)

g["class_names"] = class_names
g["class_names_level1"] = class_names_level1
g["class_names_level12"] = class_names_level12

print("\n[Three-level internal] confusion matrices →", pooled_result_dir)
plot_he_confusion_matrices(plot_confusion_matrix, pooled_result_fig, cv_data, g)
print("Saved: conf_matrix_level2 / level1 / level12 (+ L1head if available)")


## 4. StarDist predict (each dataset) — three levels

For each sample with `{sample}_matched_features_stardist.h5ad`:

Outputs under `Results/{save_result}/stardist/{sample}/`:
- Spatial: `celltype_pred_stardist_level{2,1,12}.jpg`
- ROC: `roc_stardist_level{2,1,1_L1head,12}.pdf`
- Metrics (L2): `validation_external_stardist_matched_*.csv`


### Run StarDist one level at a time

`step_pooled_stardist(pooled_ctx)` without `levels=` runs **all three heads** (L2, L1, L12).

To run a single tier, pass e.g. `levels=["l2"]`. Valid keys: `l2`, `l1`, `l12`.

- **L2 only** writes `validation_external_stardist_matched_*.csv`.
- Other tiers write spatial / ROC / AUROC table for that head.
- Optional: `samples=POOLED_SAMPLES[:1]` to test on one region first.


In [ ]:
import importlib
importlib.reload(crc_cli)

from uni_label_cv_helpers import make_pooled_stardist_result_fig

_TIER_CHECKS = {
    "l2": ("L2 celltype (final_CT)", "stardist_pred_level2", "roc_stardist_level2"),
    "l1": ("L1 lineage (final_lineage)", "stardist_pred_level1", "roc_stardist_level1"),
    "l12": ("L12 sublineage (final_sublineage)", "stardist_pred_level12", "roc_stardist_level12"),
}


def check_stardist_level(level, sample=None):
    # Print spatial/ROC file status for one tier on one sample.
    demo = sample or POOLED_SAMPLES[0]
    title, spatial_key, roc_key = _TIER_CHECKS[level.lower()]
    rf, _ = make_pooled_stardist_result_fig(DATA_ROOT, demo, save_result)
    sp, rc = rf(spatial_key), rf(roc_key)
    print(
        f"{title} ({demo}): "
        f"spatial={'OK' if Path(sp).is_file() else 'MISSING'} | "
        f"ROC={'OK' if Path(rc).is_file() else 'MISSING'}"
    )


def ensure_stardist_h5ads(samples=None, build=True):
    # Build / refresh {sample}_matched_features_stardist.h5ad when missing or
    # older than the GT StarDist CSV (e.g. after LY DCIS_1/2 rematch).
    samples = list(samples) if samples is not None else list(POOLED_SAMPLES)
    n_rebuild = 0
    for s in samples:
        p = CASES_ROOT / s / f"{s}_matched_features_stardist.h5ad"
        csv_p = CASES_ROOT / s / f"{s}_cells_matched_by_stardist.csv"
        stale = (
            p.is_file()
            and csv_p.is_file()
            and csv_p.stat().st_mtime > p.stat().st_mtime
        )
        if p.is_file() and not stale:
            continue
        n_rebuild += 1
        if not build:
            why = "STALE vs CSV" if stale else "MISSING"
            print(f"  {why} StarDist h5ad: {p}")
            continue
        print(
            f"{'Refreshing stale' if stale else 'Building'} StarDist h5ad for {s} ...",
            flush=True,
        )
        ctx = crc_cli.RunContext(
            sample=s,
            cases_root=CASES_ROOT,
            python_root=Path(path),
            therapy_data=s,
            therapy_model=crc_cli.DEFAULT_THERAPY_MODEL,
            save_result="result",
            device=device,
            seed=SEED,
            match_tolerance=1.0,
            column_rename=dict(crc_cli.HCC_COLUMN_RENAME),
            force_rebuild_h5ad=bool(stale),
            input_dim=None,
            hidden_dims=POOLED_HIDDEN_DIMS,
            cv_k=5,
            stratify_target="joint",
            patience=10,
            max_epochs=50,
            train_batch_size=4096,
            resume_from_checkpoints=True,
            ablation_tag=POOLED_ABLATION_TAG,
            hce_w1=1.0, hce_w2=2.0, hce_w12=1.0, hce_w_l12head=1.0, hce_w_l3=1.0, hce_w_l4=1.0,
            build_stardist_h5ad=True,
            acq_id=_acq_map[s],
            stardist_root=crc_cli.DEFAULT_STARDIST_ROOT,
            use_spatial_context=POOLED_USE_SPATIAL,
            spatial_k=POOLED_SPATIAL_K,
            spatial_mode=POOLED_SPATIAL_MODE,
        )
        crc_cli._ensure_stardist_h5ad(ctx)
    print(
        f"StarDist h5ad: {len(samples) - n_rebuild} up-to-date, "
        f"built/refreshed {n_rebuild}"
    )


ensure_stardist_h5ads()


### L2 celltype / final_CT (StarDist)


In [ ]:
crc_cli.step_pooled_stardist(pooled_ctx, levels=["l2"])
check_stardist_level("l2")


### L1 lineage / final_lineage (StarDist)


In [ ]:
crc_cli.step_pooled_stardist(pooled_ctx, levels=["l1"])
check_stardist_level("l1")


### L12 sublineage / final_sublineage (StarDist)


In [ ]:
crc_cli.step_pooled_stardist(pooled_ctx, levels=["l12"])
check_stardist_level("l12")


In [ ]:
# Summary check for all three tiers on first sample
print(f"StarDist outputs: {pooled_result_dir / 'stardist'}")
for level in ("l2", "l1", "l12"):
    check_stardist_level(level)


In [ ]:
# Optional: inspect L2 metrics / AUROC CSV for one sample
_demo = POOLED_SAMPLES[0]
_rf, _ = make_pooled_stardist_result_fig(DATA_ROOT, _demo, save_result)
_metrics = _rf("validation_external_stardist_matched_metrics")
_auroc = _auroc_csv_path_from_metrics(_metrics)
print(f"Demo: {_demo}")
print(" metrics:", _metrics)
print(" AUROC:", _auroc)
if Path(_metrics).is_file():
    display(pd.read_csv(_metrics).tail(1).T)


### L1 / L12 — detailed file list


In [ ]:
# Inspect L1 + L12 outputs for one sample
_demo = POOLED_SAMPLES[0]
_rf, _ = make_pooled_stardist_result_fig(DATA_ROOT, _demo, save_result)
_tiers = {
    "L1": ("stardist_pred_level1", "roc_stardist_level1", "roc_stardist_level1_L1head"),
    "L12": ("stardist_pred_level12", "stardist_true_level12", "roc_stardist_level12"),
}
for tier, keys in _tiers.items():
    print(f"\n{tier}:")
    for key in keys:
        p = Path(_rf(key))
        print(f"  {'✓' if p.is_file() else '✗'} {p.name}")


## 5. StarDist predict on all nuclei of P1/P2/P5 StarDist_Segment

Unlike **§4**, which evaluates matched GT-annotated nuclei with ground truth and AUROC, this section uses the UNI embedding to predict **every StarDist nucleus** in each CRC sample.

Build each input with:

`conda run --no-capture-output -n SeededNTM python -u code/Xenium_crc/transer_embedding_label_h5ad.py --sample {sample} --steps stardist_all_h5ad`

**Input:** `data/Xemium/CRC/Cases/{sample}/{sample}_all_features_stardist.h5ad`

**Output:** `data/Xemium/CRC/Results/{save_result}/stardist/{sample}/{sample}_all_features_stardist_label.h5ad`

The pooled CRC checkpoint predicts three heads only: `l2` (`final_CT`), `l12` (`final_sublineage`), and `l1` (`final_lineage`). The class-name order for each probability head is stored in `adata.uns['pred_prob_class_names']`.

There is no matched-ground-truth AUROC here because the prediction covers all StarDist nuclei. Spatial-context models build their neighbor graph from the StarDist centroids. Run **§2** first to create the required pooled checkpoint.

After inference, the next cells plot pred-only maps: `{sample}_stardist_pred_{l2,l12,l1}.jpg` plus an overview JPG.


```python
conda run --no-capture-output -n SeededNTM python -u \
code/Xenium_crc/transer_embedding_label_h5ad.py \
--steps stardist_all_h5ad
```


In [ ]:
# Requires pooled_ctx.g from §2 and uses the pooled CRC three-head checkpoint.
# Each sample needs {sample}_all_features_stardist.h5ad in its CRC Cases directory.
from uni_label_cv_helpers import stardist_all_label_h5ad_path

if "BEST_MLP_CHECKPOINT" not in pooled_ctx.g:
    raise RuntimeError("Run §2 (step_pooled_train) first.")

crc_cli.step_pooled_stardist_all(pooled_ctx)

print(f"StarDist-all label h5ads → {pooled_result_dir / 'stardist'}")
for sample in POOLED_SAMPLES[:5]:
    out = stardist_all_label_h5ad_path(DATA_ROOT, sample, save_result)
    if out.is_file():
        print(f"  {sample}: {out.name}")
if len(POOLED_SAMPLES) > 5:
    print(f"  ... +{len(POOLED_SAMPLES) - 5} more samples")


In [ ]:
import anndata as ad

# Inspect the first pooled CRC sample without loading the full matrix into memory.
_demo = POOLED_SAMPLES[0]
_demo_h5ad = stardist_all_label_h5ad_path(DATA_ROOT, _demo, save_result)
print(f"Demo: {_demo}\n  {_demo_h5ad}")
if _demo_h5ad.is_file():
    _a = ad.read_h5ad(_demo_h5ad, backed="r")
    try:
        prob_cols = [c for c in _a.obs.columns if "_prob_" in c]
        print(f"  n_obs={_a.n_obs:,}  prob columns={len(prob_cols)}")
        print(f"  heads in uns: {list(_a.uns.get('pred_prob_class_names', {}).keys())}")
        print(f"  sample obs cols: {list(_a.obs.columns[:8])} ...")
    finally:
        # Close the backed AnnData file handle after inspection.
        _a.file.close()
else:
    print("  (file not found — run the cell above first)")


### Plot all-nuclei predictions (L2 / L12 / L1)

Pred-only spatial maps from `{sample}_all_features_stardist_label.h5ad` (all StarDist nuclei, **no GT**).

**JPGs** (next to each label h5ad):
`Results/{save_result}/stardist/{sample}/{sample}_stardist_pred_{l2,l12,l1}.jpg`

These are distinct from §4 matched plots (`celltype_pred_stardist_level*.jpg`).


In [ ]:
# Pred-only spatial maps for every StarDist nucleus (from §5 label h5ads).
from uni_label_cv_helpers import (
    plot_stardist_label_spatial_maps,
    plot_stardist_label_spatial_overview,
    stardist_all_label_h5ad_path,
)

_ALL_HEADS = ("l2", "l12", "l1")
print(f"Plotting all-nuclei StarDist preds × {_ALL_HEADS}")
all_spatial = plot_stardist_label_spatial_maps(
    DATA_ROOT,
    POOLED_SAMPLES,
    save_result,
    label_h5ad_path_fn=stardist_all_label_h5ad_path,
    heads=_ALL_HEADS,
    pan_organ=crc_cli.PAN_ORGAN,
    show=False,
    missing_error="No all-nuclei label h5ads. Run §5 step_pooled_stardist_all first.",
)
print(f"Plotted {len(all_spatial)} sample(s).")


In [ ]:
# Compact overview: samples × L2/L12/L1
_overview_path = pooled_result_dir / "stardist" / "stardist_all_pred_overview.jpg"
plot_stardist_label_spatial_overview(
    all_spatial,
    heads=_ALL_HEADS,
    pan_organ=crc_cli.PAN_ORGAN,
    save_path=_overview_path,
    show=True,
    suptitle=f"CRC all StarDist nuclei — {save_result}",
)
print(f"Overview → {_overview_path}")


In [ ]:
## §4 batch only (matched GT nuclei + AUROC) — NOT all-nuclei.
## For all StarDist nuclei use §5: crc_cli.step_pooled_stardist_all(pooled_ctx)
# crc_cli.step_pooled_stardist(pooled_ctx, levels=["l2", "l1", "l12"])
# for level in ("l2", "l1", "l12"):
#     check_stardist_level(level)

print("Skip by default — use per-level cells in §4, or uncomment above.")


## 6. Extra StarDist-only regions

Xenium CRC has **only** annotated `P1CRC`, `P2CRC`, and `P5CRC`. There is no HCC-style rest set. Skip this section. P5 StarDist may still be missing.


In [ ]:
REST_SAMPLES_ALL = crc_cli.discover_crc_rest_samples(
    CASES_ROOT, crc_cli.DEFAULT_STARDIST_ROOT, require_h5ad=False,
)
REST_SAMPLES = crc_cli.discover_crc_rest_samples(
    CASES_ROOT, crc_cli.DEFAULT_STARDIST_ROOT, require_h5ad=True,
)
print(f"CRC rest (unannotated StarDist-only): {len(REST_SAMPLES_ALL)}")
print("Expected: 0 for the Xenium CRC P1/P2/P5.")


In [ ]:
print("No CRC rest label h5ads — annotated samples are P1CRC / P2CRC / P5CRC.")


### Plot cell type

In [ ]:
print("Skip rest spatial maps — CRC P1/P2/P5 has no unlabeled extra regions.")


In [ ]:
print("Skip rest spatial overview.")


## Notes

- CRC uses **three heads** (L2 / L12 / L1), like HCC, not lung's five-head CNiche/TNiche.
- Per-sample notebook: `CRC_train_validate_cv_UNIlabel_single.ipynb`
- Batch CLI: `CRC_train_validate_cv_UNIlabel.py --mode cross-dataset`
- `Unlabeled` is excluded from training (empty L12/L1 in the CRC `celltype` Excel sheet).
- Labels are transferred Visium HD RCTD singlet (not native Xenium GT).
